# Milestone 2: Data Processing & Transformation
**Project:** Public Health Data Visualization System  
**Dataset:** Global Health Statistics (1,000,000 records × 22 columns)  

**Objective:** Develop a robust, reproducible data processing pipeline that cleans, transforms, and engineers features from the raw dataset.

---
**Assigned to:** *(Ritah, Sharon, Joyce, Christine, Jubilee)*  
**Branch:** `feature/milestone2-pipeline`

> **Before starting:** Run `load_dataset.ipynb` to confirm the dataset is available locally.

---
## 1. Setup & Data Loading

**What to do here:**
- Import all required libraries (`pandas`, `numpy`, `os`, `pathlib`, etc.)
- Load the raw dataset from `data/raw/` using a reproducible file path
- Print the shape and a preview of the data to confirm it loaded correctly

**Deliverable check:** The pipeline must start from a reproducible data source — no hardcoded absolute paths.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolving project root and defining file paths
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'Global Health Statistics.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset path : {RAW_DATA}')
print(f'Dataset exists: {RAW_DATA.exists()}')

df = pd.read_csv(RAW_DATA)
print(f'\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Project root : /home/mendarrr/Projects/Data Science/Python/Data-Visualization-Project
Dataset path : /home/mendarrr/Projects/Data Science/Python/Data-Visualization-Project/data/raw/Global Health Statistics.csv
Dataset exists: True

Shape: 1,000,000 rows x 22 columns


,Country,Year,Disease Name,Disease Category,Prevalence Rate (%),Incidence Rate (%),Mortality Rate (%),Age Group,Gender,Population Affected,...,Hospital Beds per 1000,Treatment Type,Average Treatment Cost (USD),Availability of Vaccines/Treatment,Recovery Rate (%),DALYs,Improvement in 5 Years (%),Per Capita Income (USD),Education Index,Urbanization Rate (%)
0,Italy,2013,Malaria,Respiratory,0.95,1.55,8.42,0-18,Male,471007,...,7.58,Medication,21064,No,91.82,4493,2.16,16886,0.79,86.02
1,France,2002,Ebola,Parasitic,12.46,8.63,8.75,61+,Male,634318,...,5.11,Surgery,47851,Yes,76.65,2366,4.82,80639,0.74,45.52
2,Turkey,2015,COVID-19,Genetic,0.91,2.35,6.22,36-60,Male,154878,...,3.49,Vaccination,27834,Yes,98.55,41,5.81,12245,0.41,40.20
3,Indonesia,2011,Parkinson's Disease,Autoimmune,4.68,6.29,3.99,0-18,Other,446224,...,8.44,Surgery,144,Yes,67.35,3201,2.22,49336,0.49,58.47
4,Italy,2013,Tuberculosis,Genetic,0.83,13.59,7.01,61+,Male,472908,...,5.90,Medication,8908,Yes,50.06,2832,6.93,47701,0.50,48.14


---
## 2. Data Quality Audit

**What to do here:**
- Scan for **missing values** — count and percentage per column
- Detect **duplicate rows**
- Identify **statistical anomalies/outliers** in numerical columns (e.g., using IQR or z-score)
- Check for **invalid values** (e.g., percentages outside 0–100, negative values where impossible)
- Check **data types** — are columns stored as the correct type?

**Deliverable check:** This section documents all issues *before* they are fixed. Think of it as a diagnostic report.

In [2]:
# Check for missing values (count + percentage per column)
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Missing_Percentage': missing_percentage.values
}).sort_values('Missing_Count', ascending=False)

print("Missing Values Report:")
print(missing_df)
print(f"\nTotal rows with at least one missing value: {df.isnull().any(axis=1).sum()}")

Missing Values Report:
                                Column  Missing_Count  Missing_Percentage
0                              Country              0                 0.0
1                                 Year              0                 0.0
20                     Education Index              0                 0.0
19             Per Capita Income (USD)              0                 0.0
18          Improvement in 5 Years (%)              0                 0.0
17                               DALYs              0                 0.0
16                   Recovery Rate (%)              0                 0.0
15  Availability of Vaccines/Treatment              0                 0.0
14        Average Treatment Cost (USD)              0                 0.0
13                      Treatment Type              0                 0.0
12              Hospital Beds per 1000              0                 0.0
11                    Doctors per 1000              0                 0.0
10             

In [3]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")
print(f"Unique rows: {len(df) - duplicates}")
print(f"Duplicate ratio: {(duplicates / len(df) * 100):.2f}%")

# Show duplicate rows if any exist
if duplicates > 0:
    print("\nSample duplicate rows:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
else:
    print("\nNo duplicate rows found.")

Total duplicate rows: 0
Unique rows: 1000000
Duplicate ratio: 0.00%

No duplicate rows found.


In [4]:
# Check data types and flag any mismatches
print("Current Data Types:")
print(df.dtypes)
print("\n" + "="*60)

# Identify potential type mismatches
dtype_issues = []

for col in df.columns:
    # Check if numeric columns are stored as object
    if 'rate' in col.lower() or 'percentage' in col.lower() or 'population' in col.lower():
        if df[col].dtype == 'object':
            dtype_issues.append(f"'{col}' — should be numeric but is {df[col].dtype}")
    
    # Check if year columns are numeric
    if 'year' in col.lower():
        if df[col].dtype != 'int64' and df[col].dtype != 'float64':
            dtype_issues.append(f"'{col}' — should be numeric but is {df[col].dtype}")
    
    # Check for date-like columns stored as strings
    if 'date' in col.lower():
        if df[col].dtype == 'object':
            dtype_issues.append(f"'{col}' — should be datetime but is {df[col].dtype}")

print("\nData Type Issues Found:")
if dtype_issues:
    for issue in dtype_issues:
        print(f"  ⚠ {issue}")
else:
    print("  ✓ No obvious type mismatches detected.")

# Show memory usage by dtype
print("\n" + "="*60)
print("Memory Usage by Data Type:")
print(df.memory_usage(deep=True) / 1024**2)  # in MB

Current Data Types:
Country                                object
Year                                    int64
Disease Name                           object
Disease Category                       object
Prevalence Rate (%)                   float64
Incidence Rate (%)                    float64
Mortality Rate (%)                    float64
Age Group                              object
Gender                                 object
Population Affected                     int64
Healthcare Access (%)                 float64
Doctors per 1000                      float64
Hospital Beds per 1000                float64
Treatment Type                         object
Average Treatment Cost (USD)            int64
Availability of Vaccines/Treatment     object
Recovery Rate (%)                     float64
DALYs                                   int64
Improvement in 5 Years (%)            float64
Per Capita Income (USD)                 int64
Education Index                       float64
Urbanization R

In [25]:
# Detect outliers/anomalies in numerical columns (IQR or z-score method)
from scipy import stats

# Get numerical columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns

print("OUTLIER DETECTION — IQR Method")
print("="* 150)

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    
    if outlier_count > 0:
        outlier_summary.append({
            'Column': col,
            'Outlier_Count': outlier_count,
            'Outlier_Percentage': outlier_pct,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_Count', ascending=False)
    print(outlier_df.to_string(index=False))
else:
    print("✓ No outliers detected using IQR method.")

# Z-score method (values with |z-score| > 3 are extreme outliers)
print("\n" + "="* 150)
print("OUTLIER DETECTION — Z-Score Method (|z| > 3)")
print("="* 150)

z_outliers = []
for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    extreme_outliers = (z_scores > 3).sum()
    if extreme_outliers > 0:
        z_outliers.append({
            'Column': col,
            'Extreme_Outliers_(z>3)': extreme_outliers,
            'Percentage': (extreme_outliers / len(df)) * 100
        })

if z_outliers:
    z_df = pd.DataFrame(z_outliers).sort_values('Extreme_Outliers_(z>3)', ascending=False)
    print(z_df.to_string(index=False))
else:
    print("✓ No extreme outliers detected using z-score method.")

OUTLIER DETECTION — IQR Method
✓ No outliers detected using IQR method.

OUTLIER DETECTION — Z-Score Method (|z| > 3)
✓ No extreme outliers detected using z-score method.


In [27]:
# Check for invalid values (e.g., rates outside 0-100, negative populations)
print("INVALID VALUES CHECK")
print("=" * 150)

invalid_issues = []

for col in df.columns:
    # Check for negative values in columns that should be non-negative (only numeric columns)
    if ('population' in col.lower() or 'count' in col.lower() or 'cases' in col.lower()) and df[col].dtype in ['float64', 'int64']:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Negative values (should be ≥0)',
                'Count': negative_count,
                'Percentage': (negative_count / len(df)) * 100
            })
    
    # Check for percentages/rates outside 0-100 range
    if ('rate' in col.lower() or 'percentage' in col.lower() or '(%)' in col) and df[col].dtype in ['float64', 'int64']:
        below_0 = (df[col] < 0).sum()
        above_100 = (df[col] > 100).sum()
        
        if below_0 > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Below 0 (should be 0-100)',
                'Count': below_0,
                'Percentage': (below_0 / len(df)) * 100
            })
        if above_100 > 0:
            invalid_issues.append({
                'Column': col,
                'Issue': 'Above 100 (should be 0-100)',
                'Count': above_100,
                'Percentage': (above_100 / len(df)) * 100
            })

if invalid_issues:
    invalid_df = pd.DataFrame(invalid_issues).sort_values('Count', ascending=False)
    print(invalid_df.to_string(index=False))
else:
    print("✓ No invalid values detected.")

# Show sample statistics for key numeric columns
print("\n" + "=" * 150)
print("Numeric Column Statistics (Quick Check):")
print("="* 150)
numeric_stats = df.select_dtypes(include=[np.number]).describe().loc[['min', 'max']].T
print(numeric_stats)

INVALID VALUES CHECK
✓ No invalid values detected.

Numeric Column Statistics (Quick Check):
                                 min        max
Year                          2000.0     2024.0
Prevalence Rate (%)              0.1       20.0
Incidence Rate (%)               0.1       15.0
Mortality Rate (%)               0.1       10.0
Population Affected           1000.0  1000000.0
Healthcare Access (%)           50.0      100.0
Doctors per 1000                 0.5        5.0
Hospital Beds per 1000           0.5       10.0
Average Treatment Cost (USD)   100.0    50000.0
Recovery Rate (%)               50.0       99.0
DALYs                            1.0     5000.0
Improvement in 5 Years (%)       0.0       10.0
Per Capita Income (USD)        500.0   100000.0
Education Index                  0.4        0.9
Urbanization Rate (%)           20.0       90.0


**Audit Summary** *(fill in after running the cells above)*

| Issue | Column(s) Affected | Count | Action Planned |
|---|---|---|---|
| Missing values | | | |
| Duplicate rows | — | | |
| Outliers | | | |
| Invalid values | | | |
| Type mismatches | | | |

---
## 3. Data Cleaning & Standardization

**What to do here:**
- Handle **missing values** (impute, drop, or flag — justify your choice)
- Remove or correct **duplicate rows**
- Handle **outliers** (cap, remove, or transform — justify your choice)
- Fix **data type mismatches** (e.g., convert columns to correct types)
- Standardize **string/categorical values** (e.g., consistent casing, strip whitespace)
- Correct any **invalid values** found in the audit

**Deliverable check:** Each cleaning step should be a reusable function where possible, making the pipeline reproducible.

In [28]:
# Handle missing values
# Strategy:
#   - Numeric columns: impute with median (robust to outliers)
#   - Categorical columns: impute with mode (most frequent value)
#   - Create indicator columns for rows that had missing values (for feature engineering)

df_cleaned = df.copy()

# Track which rows had missing values
df_cleaned['Had_Missing_Values'] = df.isnull().any(axis=1).astype(int)

print("Missing Values Handling:")
print("="* 150)

# Identify columns with missing values
cols_with_missing = df_cleaned.isnull().sum()
cols_with_missing = cols_with_missing[cols_with_missing > 0]

if len(cols_with_missing) > 0:
    print(f"Columns with missing values: {list(cols_with_missing.index)}")
    
    for col in cols_with_missing.index:
        if df_cleaned[col].dtype in ['float64', 'int64']:
            # Numeric: impute with median
            median_val = df_cleaned[col].median()
            df_cleaned[col].fillna(median_val, inplace=True)
            print(f"  • {col}: Imputed {cols_with_missing[col]} values with median ({median_val:.2f})")
        else:
            # Categorical: impute with mode
            mode_val = df_cleaned[col].mode()[0]
            df_cleaned[col].fillna(mode_val, inplace=True)
            print(f"  • {col}: Imputed {cols_with_missing[col]} values with mode ('{mode_val}')")
else:
    print("✓ No missing values found — no imputation needed.")

print("\n" + "=" * 150)
print(f"Rows with missing values flag: {df_cleaned['Had_Missing_Values'].sum()}")
print(f"Remaining missing values in dataset: {df_cleaned.isnull().sum().sum()}")


Missing Values Handling:
✓ No missing values found — no imputation needed.

Rows with missing values flag: 0
Remaining missing values in dataset: 0


In [29]:
# Remove duplicate rows
print("Duplicate Rows Removal:")
print("=" * 150)

rows_before = len(df_cleaned)

# Remove exact duplicates (all columns match)
df_cleaned = df_cleaned.drop_duplicates()

rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after

print(f"Rows before deduplication: {rows_before:,}")
print(f"Rows after deduplication: {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")
print(f"Duplicate ratio: {(rows_removed / rows_before * 100):.2f}%")

if rows_removed > 0:
    print(f"\n✓ Successfully removed {rows_removed:,} duplicate rows.")
else:
    print("\n✓ No duplicate rows found.")

Duplicate Rows Removal:
Rows before deduplication: 1,000,000
Rows after deduplication: 1,000,000
Duplicate rows removed: 0
Duplicate ratio: 0.00%

✓ No duplicate rows found.


In [30]:
# Handle outliers
# Approach: Cap at IQR bounds (preserves data volume while reducing extreme values)
# Justification: Capping is preferred over removal to maintain sample size; more robust than winsorization for extreme outliers

print("Outlier Handling:")
print("=" * 150)

# Track rows with outliers
df_cleaned['Had_Outliers'] = False

numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
outlier_caps_applied = []

for col in numeric_cols:
    # Skip the indicator columns we added
    if col in ['Had_Missing_Values', 'Had_Outliers']:
        continue
    
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers before capping
    outliers_below = (df_cleaned[col] < lower_bound).sum()
    outliers_above = (df_cleaned[col] > upper_bound).sum()
    
    if outliers_below > 0 or outliers_above > 0:
        # Mark rows with outliers
        df_cleaned.loc[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound), 'Had_Outliers'] = True
        
        # Cap values at bounds
        df_cleaned[col] = df_cleaned[col].clip(lower_bound, upper_bound)
        
        outlier_caps_applied.append({
            'Column': col,
            'Below_Lower_Bound': outliers_below,
            'Above_Upper_Bound': outliers_above,
            'Total_Capped': outliers_below + outliers_above,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })

if outlier_caps_applied:
    outlier_caps_df = pd.DataFrame(outlier_caps_applied)
    print(outlier_caps_df.to_string(index=False))
    print(f"\n✓ Successfully capped {outlier_caps_df['Total_Capped'].sum():,} outlier values across {len(outlier_caps_df)} columns.")
else:
    print("✓ No outliers detected — no capping needed.")

print("=" * 150)
print(f"Rows with outliers capped: {df_cleaned['Had_Outliers'].sum():,}")

Outlier Handling:
✓ No outliers detected — no capping needed.
Rows with outliers capped: 0


In [31]:
# Fix data types (e.g., Year to int, categorical columns to category dtype)
print("Data Type Conversion:")
print("=" * 150)

dtypes_before = df_cleaned.dtypes.copy()

# Convert Year to int
if 'Year' in df_cleaned.columns:
    df_cleaned['Year'] = df_cleaned['Year'].astype('int64')
    print("✓ Year converted to int64")

# Identify and convert categorical columns (object dtype columns with high cardinality are typically categorical)
categorical_cols = df_cleaned.select_dtypes(include='object').columns.tolist()

# Remove indicator columns from categorical conversion
categorical_cols = [col for col in categorical_cols if col not in ['Had_Missing_Values', 'Had_Outliers']]

conversions = []
for col in categorical_cols:
    unique_count = df_cleaned[col].nunique()
    total_rows = len(df_cleaned)
    uniqueness_pct = (unique_count / total_rows) * 100
    
    # Convert to category if unique values < 20% of total rows (typical threshold for categorical data)
    if uniqueness_pct < 20:
        df_cleaned[col] = df_cleaned[col].astype('category')
        conversions.append({
            'Column': col,
            'Original_Type': 'object',
            'New_Type': 'category',
            'Unique_Values': unique_count
        })

if conversions:
    conversions_df = pd.DataFrame(conversions)
    print("\nCategorical Columns Converted:")
    print(conversions_df.to_string(index=False))
else:
    print("\nNo columns eligible for category conversion (high cardinality).")

# Show memory usage before and after
print("\n" + "=" * 150)
print("Memory Usage Comparison:")
dtypes_after = df_cleaned.dtypes
memory_before = df.memory_usage(deep=True).sum() / 1024**2
memory_after = df_cleaned.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
memory_saved_pct = (memory_saved / memory_before) * 100

print(f"Before optimization: {memory_before:.2f} MB")
print(f"After optimization:  {memory_after:.2f} MB")
print(f"Memory saved:        {memory_saved:.2f} MB ({memory_saved_pct:.1f}%)")

print("\n" + "=" * 150)
print("Final Data Types:")
print(df_cleaned.dtypes)

Data Type Conversion:
✓ Year converted to int64

Categorical Columns Converted:
                            Column Original_Type New_Type  Unique_Values
                           Country        object category             20
                      Disease Name        object category             20
                  Disease Category        object category             11
                         Age Group        object category              4
                            Gender        object category              3
                    Treatment Type        object category              4
Availability of Vaccines/Treatment        object category              2

Memory Usage Comparison:
Before optimization: 484.66 MB
After optimization:  129.71 MB
Memory saved:        354.96 MB (73.2%)

Final Data Types:
Country                               category
Year                                     int64
Disease Name                          category
Disease Category                      category
Pr

In [11]:
# Standardize string/categorical values (strip whitespace, consistent casing)


In [12]:
# Verify cleaning results — rerun the audit checks and confirm issues are resolved


---
## 4. Transformation: Joins, Grouping & Aggregations

**What to do here:**
- Create **grouped summaries** (e.g., average mortality rate by country, by disease category, by year) => Like those in the CATs
- Perform **aggregations** that will be useful for visualization (e.g., total population affected per region/year)
- If working with multiple data sources, perform **joins/merges** here
- Create any **pivot tables** or **cross-tabulations** that reveal patterns

**Deliverable check:** Covers the "Transformation (aggregation, grouping, joins)" requirement.

In [13]:
# Group by Country — compute mean Mortality Rate, Recovery Rate, Healthcare Access


In [14]:
# Group by Disease Category — compute summary statistics


In [15]:
# Group by Year — compute trends over time


In [16]:
# Any joins/merges with additional data sources (if applicable)


In [17]:
# Pivot table or cross-tabulation (e.g., Disease Category x Age Group)


---
## 5. Feature Engineering

**What to do here:**
- Create **new derived columns** that add analytical value, for example:
  - `Mortality_to_Prevalence_Ratio` = Mortality Rate / Prevalence Rate
  - `Healthcare_Gap` = 100 - Healthcare Access (%)
  - `High_Mortality_Flag` = 1 if Mortality Rate > threshold, else 0
  - `Decade` = Year grouped into decades (2000s, 2010s, 2020s)
  - Encoded versions of categorical columns (label encoding or one-hot)
- Document **why** each feature was created and what it represents

**Deliverable check:** Explicitly addresses the "Feature engineering" requirement.

In [18]:
# Create ratio/derived numerical features
# Example: df['Mortality_to_Prevalence_Ratio'] = df['Mortality Rate (%)'] / df['Prevalence Rate (%)']


In [19]:
# Create binary flag features (e.g., High_Mortality_Flag, Vaccine_Available_Flag)


In [20]:
# Create time-based features (e.g., Decade, Years_Since_2000)


In [21]:
# Encode categorical columns for future modeling (label encoding or one-hot)


In [22]:
# Preview the final dataframe with all new features


---
## 6. Final Pipeline & Dataset Export

**What to do here:**
- Print a final summary of the cleaned + engineered dataset (shape, columns, dtypes)
- Export the processed dataset to `data/processed/` as a CSV file
- Write a **Data Dictionary** table below documenting every transformation made

**Deliverable check:** Delivers the "Clean and structured dataset" and "Documentation of transformations".

In [23]:
# Print final dataset summary (shape, dtypes, head)


In [24]:
# Export cleaned dataset to data/processed/
# Example:
# PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'global_health_cleaned.csv'
# df_clean.to_csv(PROCESSED_PATH, index=False)
# print(f"Saved to {PROCESSED_PATH}")


### Data Dictionary — Transformations Applied

*(Fill in after completing all sections above)*

| Column / Feature | Type | Description | Transformation Applied |
|---|---|---|---|
| *(original column)* | *(dtype)* | *(what it represents)* | *(e.g., no change / imputed mean / type cast)* |
| *(new feature)* | *(dtype)* | *(what it represents)* | *(e.g., derived from X / Y)* |

### Pipeline Summary

*(Fill in after completing all sections above)*

| Step | Action | Rows Before | Rows After | Columns Before | Columns After |
|---|---|---|---|---|---|
| Raw load | — | | | | |
| Remove duplicates | | | | | |
| Handle missing values | | | | | |
| Handle outliers | | | | | |
| Feature engineering | | | | | |
| **Final export** | | | | | |